# Fine-tune DistilBERT for Contradiction Detection
Upload `conflictbank_small.jsonl` (3.8MB) when prompted.
Expected training time: 10-15 minutes on T4 GPU.
Expected results: temporal F1~0.99, semantic F1~0.75-0.85, misinfo F1~0.70-0.80

In [ ]:
#@title Check GPU
!nvidia-smi

In [ ]:
#@title Install dependencies
!pip install transformers torch scikit-learn -q

In [ ]:
#@title Upload data
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
#@title Load data
import json
train_data = []
val_data = []
test_data = []

with open('conflictbank_small.jsonl', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        text = item['claim'] + ' [SEP] ' + item['evidence']
        entry = {'text': text, 'label': item['label']}
        if item['split'] == 'train':
            train_data.append(entry)
        elif item['split'] == 'val':
            val_data.append(entry)
        else:
            test_data.append(entry)

print(f'Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')
from collections import Counter
print('Train dist:', Counter(d['label'] for d in train_data))

In [ ]:
#@title Tokenize
from transformers import DistilBertTokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(data):
    texts = [d['text'] for d in data]
    enc = tokenizer(texts, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    return enc, [d['label'] for d in data]

train_enc, train_labels = tokenize(train_data)
val_enc, val_labels = tokenize(val_data)
test_enc, test_labels = tokenize(test_data)
print('Tokenization done')

In [ ]:
#@title Train model
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import DistilBertForSequenceClassification, AdamW
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)
model.to(device)

train_dataset = TensorDataset(train_enc['input_ids'], train_enc['attention_mask'], torch.tensor(train_labels))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()
for epoch in range(3):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}: avg loss = {total_loss/len(train_loader):.4f}')

print('Training complete!')

In [ ]:
#@title Evaluate
from sklearn.metrics import classification_report, f1_score
import numpy as np

model.eval()
test_dataset = TensorDataset(test_enc['input_ids'], test_enc['attention_mask'])
test_loader = DataLoader(test_dataset, batch_size=32)

all_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating'):
        input_ids, attention_mask = [b.to(device) for b in batch]
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)

y_test = np.array(test_labels)
y_pred = np.array(all_preds)

print(f'Test F1(macro) = {f1_score(y_test, y_pred, average="macro"):.4f}')
print(classification_report(y_test, y_pred, target_names=['default','temporal','semantic','misinfo'], digits=3))

# Binary
y_test_bin = (y_test > 0).astype(int)
y_pred_bin = (y_pred > 0).astype(int)
from sklearn.metrics import precision_score, recall_score
print(f'Binary: P={precision_score(y_test_bin, y_pred_bin):.3f} R={recall_score(y_test_bin, y_pred_bin):.3f} F1={f1_score(y_test_bin, y_pred_bin):.3f}')

In [ ]:
#@title Download model
model.save_pretrained('contradiction_distilbert')
tokenizer.save_pretrained('contradiction_distilbert')
!zip -r contradiction_distilbert.zip contradiction_distilbert/
from google.colab import files
files.download('contradiction_distilbert.zip')
print('Download started!')